# Feature Engineering 

This notebook constructs meaningful features from raw GPS coordinates
to improve model performance.

Covered steps :
1. Loads train / val / test splits
2. Engineers all necessary features
3. Encodes categorical variables
4. Drops unnecessary columns
5. Exports `train_ready.csv`, `val_ready.csv`, `test_ready.csv`

## 1. Setup and Load Data

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

TARGET_COL = 'target_label'

train = pd.read_csv('../data/processed/train.csv')
val   = pd.read_csv('../data/processed/val.csv')
test  = pd.read_csv('../data/processed/test.csv')

for name, split in [('train', train), ('val', val), ('test', test)]:
    print(f"{name}: {split.shape}")
print(f'\nColumns: {list(train.columns)}')

train: (4664, 25)
val: (999, 25)
test: (1000, 25)

Columns: ['id', 'year', 'section', 'user', 'amphitheatre', 'module', 'seat_block', 'seat_row', 'seat_column', 'latitude_mean', 'longitude_mean', 'accuracy_mean', 'gps_variance', 'is_outside', 'sample_count', 'raw_gps_readings', 'collection_metadata', 'navigator_context', 'screen_context', 'network_information', 'battery_status', 'timestamp', 'created_at', 'device_info', 'target_label']


##  2. Drop Unnecessary Columns

In [2]:
columns_to_drop = [
    'id',
    'user',
    'year',
    'section',
    'amphitheatre',
    'module',
    'navigator_context',
    'screen_context',
    'network_information',
    'battery_status',
    'device_info',
    'raw_gps_readings',
    'collection_metadata',
    'created_at',
]

def drop_cols(df):
    existing = [c for c in columns_to_drop if c in df.columns]
    return df.drop(columns=existing)

train = drop_cols(train)
val   = drop_cols(val)
test  = drop_cols(test)

print(f"train shape after drop: {train.shape}")
print(f"val   shape after drop: {val.shape}")
print(f"test  shape after drop: {test.shape}")
print(f"\nRemaining columns:")
for col in sorted(train.columns):
    print(f"  - {col}")

train shape after drop: (4664, 11)
val   shape after drop: (999, 11)
test  shape after drop: (1000, 11)

Remaining columns:
  - accuracy_mean
  - gps_variance
  - is_outside
  - latitude_mean
  - longitude_mean
  - sample_count
  - seat_block
  - seat_column
  - seat_row
  - target_label
  - timestamp


## 3. Amphitheatre centroids (computed from training set only)

We use the **mean GPS position per amphitheatre computed from the training set** as reference centroids.
Computing from train only prevents any leakage into val/test.

These centroids are then applied to all three splits to compute distance-based features.

In [3]:
AMPHI_LABELS = [f'Amphi {i}' for i in range(1, 9)]

centroids = (
    train[train[TARGET_COL].isin(AMPHI_LABELS)]
    .groupby(TARGET_COL)[['latitude_mean', 'longitude_mean']]
    .mean()
    .rename(columns={'latitude_mean': 'c_lat', 'longitude_mean': 'c_lon'})
)
print('Centroids (from train set):')
print(centroids.round(6).to_string())

Centroids (from train set):
                  c_lat     c_lon
target_label                     
Amphi 1       36.688396  2.866703
Amphi 2       36.688353  2.866493
Amphi 3       36.688389  2.866236
Amphi 4       36.688306  2.866123
Amphi 5       36.688361  2.866690
Amphi 6       36.688351  2.866476
Amphi 7       36.688358  2.866284
Amphi 8       36.688336  2.866116


## 4. Distance features

For each sample we compute the **Euclidean distance** (in degrees, fine at this scale) to every
amphitheatre centroid. This gives the model spatial context that raw lat/lon can't express alone.

| Feature | Intuition |
|---|---|
| `dist_Amphi_N` | How far the GPS reading is from each known amphitheatre |
| `dist_nearest` | Distance to the closest amphitheatre |
| `dist_2nd` | Distance to the 2nd closest — gap between 1st and 2nd signals ambiguity |
| `dist_gap` | `dist_2nd − dist_nearest`: large gap = confident placement, small gap = confused |


> **Why Euclidean and not Haversine?**  
> At this scale (~100 m campus), Euclidean error vs Haversine is < 0.01 m. Not worth the overhead.

In [4]:
def add_distance_features(df, centroids):
    df = df.copy()
    dist_cols = []

    for lbl, row in centroids.iterrows():
        col = f'dist_{lbl.replace(" ", "_")}'
        df[col] = np.sqrt(
            (df['latitude_mean'] - row['c_lat'])**2 +
            (df['longitude_mean'] - row['c_lon'])**2
        )
        dist_cols.append(col)

    dist_matrix = df[dist_cols]

    df['dist_nearest'] = dist_matrix.min(axis=1)
    df['dist_2nd'] = dist_matrix.apply(lambda r: r.nsmallest(2).iloc[-1], axis=1)
    df['dist_gap'] = df['dist_2nd'] - df['dist_nearest']

    df['nearest_amphi'] = dist_matrix.idxmin(axis=1).str.replace('dist_', '').str.replace('_', ' ')

    le_nearest = LabelEncoder()
    le_nearest.fit(df['nearest_amphi'])
    df['nearest_amphi_enc'] = le_nearest.transform(df['nearest_amphi'])

    return df, dist_cols, le_nearest

train, dist_cols, le_nearest = add_distance_features(train, centroids)
val,   _,         _          = add_distance_features(val,   centroids)
test,  _,         _          = add_distance_features(test,  centroids)

val['nearest_amphi_enc']  = le_nearest.transform(val['nearest_amphi'])
test['nearest_amphi_enc'] = le_nearest.transform(test['nearest_amphi'])

print(f'Distance features added: {len(dist_cols)} centroid distances + 3 summary features + 1 nearest_amphi')
print(f'\nNearest amphitheatre mapping: {dict(zip(le_nearest.classes_, le_nearest.transform(le_nearest.classes_)))}')
print(f'\nFirst few rows:')
train[dist_cols + ['dist_nearest', 'dist_gap', 'nearest_amphi', 'nearest_amphi_enc']].head(5)

Distance features added: 8 centroid distances + 3 summary features + 1 nearest_amphi

Nearest amphitheatre mapping: {'Amphi 1': np.int64(0), 'Amphi 2': np.int64(1), 'Amphi 3': np.int64(2), 'Amphi 4': np.int64(3), 'Amphi 5': np.int64(4), 'Amphi 6': np.int64(5), 'Amphi 7': np.int64(6), 'Amphi 8': np.int64(7)}

First few rows:


,dist_Amphi_1,dist_Amphi_2,dist_Amphi_3,dist_Amphi_4,dist_Amphi_5,dist_Amphi_6,dist_Amphi_7,dist_Amphi_8,dist_nearest,dist_gap,nearest_amphi,nearest_amphi_enc
0,0.000598,0.000383,0.000164,0.000032,0.000578,0.000366,0.000186,0.000061,0.000032,0.000029,Amphi 4,3
1,0.000211,0.000009,0.000266,0.000377,0.000192,0.000023,0.000214,0.000383,0.000009,0.000014,Amphi 2,1
2,0.000233,0.000021,0.000246,0.000355,0.000214,0.000011,0.000193,0.000361,0.000011,0.000010,Amphi 6,5
3,0.000204,0.000013,0.000272,0.000384,0.000185,0.000029,0.000221,0.000390,0.000013,0.000016,Amphi 2,1
4,0.000226,0.000013,0.000247,0.000361,0.000209,0.000006,0.000196,0.000365,0.000006,0.000007,Amphi 6,5


## 5. GPS quality features

The GPS columns `accuracy_mean` and `gps_variance` are **diagnostic** signals.

**Important:** `gps_variance` is **constant 0** for this entire dataset — it was crushed
to zero by the IQR outlier filter in preprocessing (99th percentile cap = 0).
We therefore **drop** `log_variance` and `sample_count_clipped` from the feature set.

| Feature | Intuition |
|---|---|
| `log_accuracy` | Log-transform compresses the heavy right tail of accuracy values |
| `accuracy_bin` | Discretised accuracy (good / ok / bad) — captures non-linear threshold effects |
| `high_accuracy_flag` | Binary: accuracy < 30 m (very good fix) — Outside readings average 9 m, Amphi 1/7 average 24 m |

In [5]:
def add_gps_quality_features(df, median_accuracy=None):
    df = df.copy()
    if median_accuracy is None:
        median_accuracy = df['accuracy_mean'].median()
    df['log_accuracy'] = np.log1p(df['accuracy_mean'].fillna(median_accuracy))
    df['accuracy_bin'] = pd.cut(
        df['accuracy_mean'].fillna(999),
        bins=[0, 30, 80, 999],
        labels=[0, 1, 2]
    ).astype(int)
    df['high_accuracy_flag'] = (df['accuracy_mean'] < 30).astype(int)
    return df, median_accuracy

train, median_accuracy = add_gps_quality_features(train)
val,   _               = add_gps_quality_features(val,  median_accuracy)
test,  _               = add_gps_quality_features(test, median_accuracy)

print('GPS quality features added.')
train[['log_accuracy', 'accuracy_bin', 'high_accuracy_flag']].describe()

GPS quality features added.


,log_accuracy,accuracy_bin,high_accuracy_flag
count,4664.000000,4664.000000,4664.000000
mean,2.908039,0.090051,0.909949
std,0.404167,0.286286,0.286286
min,1.606634,0.000000,0.000000
25%,2.681569,0.000000,1.000000
50%,2.932207,0.000000,1.000000
75%,3.197618,0.000000,1.000000
max,3.946444,1.000000,1.000000


## 6. Seat zone features

When available, `seat_block`, `seat_row`, `seat_column` tell us exactly where in an amphitheatre
the student sat. This is extremely discriminative - row 1 of Amphi 4 has a very specific GPS cloud.

About 437 rows are missing seat info (outside / campus-wide recordings).  
We encode these as a dedicated category and add a `has_seat` flag.

In [6]:
def add_seat_features(df):
    df = df.copy()
    df['has_seat'] = df['seat_row'].notna().astype(int)
    block_map = {'Left': 0, 'Center': 1, 'Right': 2}
    df['seat_block_enc'] = df['seat_block'].map(block_map).fillna(3).astype(int)
    df['seat_row_filled']    = df['seat_row'].fillna(-1)
    df['seat_column_filled'] = df['seat_column'].fillna(-1)
    def zone_id(row):
        if row['has_seat'] == 0:
            return -1
        return int(row['seat_row_filled']) * 100 + int(row['seat_column_filled'])
    df['seat_zone_id'] = df.apply(zone_id, axis=1)
    return df

train = add_seat_features(train)
val   = add_seat_features(val)
test  = add_seat_features(test)

print(f"Rows with seat info — train: {train['has_seat'].sum()} / {len(train)} ({train['has_seat'].mean():.1%})")
train[['has_seat','seat_block_enc','seat_row_filled','seat_column_filled']].value_counts(['has_seat']).to_frame()

Rows with seat info — train: 4227 / 4664 (90.6%)


,count
has_seat,
1,4227
0,437


## 7. Temporal features

**Important caveat:** Inspecting the data reveals that `day_of_week` encodes the
*data collection schedule*, not real-world signal:
- Amphi 4, 5, 6, 7 were collected exclusively on Sundays (day_of_week = 6)
- Amphi 8 was collected only on Mon/Wed

Using `day_of_week` or `is_weekend` would cause the model to memorise collection dates
and **fail completely in production**. These are therefore **excluded**.

We keep only **hour-of-day** features with cyclic encoding, which capture genuine
lecture schedule patterns that will be stable at inference time.

| Feature | Intuition |
|---|---|
| `hour_of_day` | Raw hour (kept for interpretability) |
| `hour_sin` / `hour_cos` | Cyclic encoding — avoids 23→0 discontinuity |

In [7]:
def add_temporal_features(df):
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
    ts = df['timestamp']
    df['hour_of_day'] = ts.dt.hour
    df['hour_sin'] = np.sin(2 * np.pi * df['hour_of_day'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour_of_day'] / 24)
    return df

train = add_temporal_features(train)
val   = add_temporal_features(val)
test  = add_temporal_features(test)

print('Temporal features added (hour only — day/weekend excluded).')
train[['hour_of_day', 'hour_sin', 'hour_cos']].describe()

Temporal features added (hour only — day/weekend excluded).


,hour_of_day,hour_sin,hour_cos
count,4664.000000,4.664000e+03,4664.000000
mean,10.793525,3.139720e-01,-0.593449
std,3.529605,6.559706e-01,0.345052
min,7.000000,-8.660254e-01,-1.000000
25%,9.000000,1.224647e-16,-0.707107
50%,9.000000,7.071068e-01,-0.707107
75%,12.000000,7.071068e-01,-0.500000
max,20.000000,9.659258e-01,0.500000


## 8. Label encoding

Convert string labels -> integers (required by sklearn / PyTorch).  
We fit the encoder on train only and apply to val and test.

In [8]:
le = LabelEncoder()
le.fit(train[TARGET_COL])

train['label_enc'] = le.transform(train[TARGET_COL])
val['label_enc']   = le.transform(val[TARGET_COL])
test['label_enc']  = le.transform(test[TARGET_COL])

label_map = {str(cls): int(idx) for idx, cls in enumerate(le.classes_)}
print('Label mapping:', label_map)

Label mapping: {'Amphi 1': 0, 'Amphi 2': 1, 'Amphi 3': 2, 'Amphi 4': 3, 'Amphi 5': 4, 'Amphi 6': 5, 'Amphi 7': 6, 'Amphi 8': 7, 'Outside': 8}


## 9. Dropping Redundant Columns After Encoding

After engineering features and encoding categorical variables, we have several redundant columns that should be removed.

In [9]:
columns_to_drop_final = [
    'timestamp',
    'seat_block',
    'latitude_mean',
    'longitude_mean',
    'gps_variance',
    'target_label',
    'hour_of_day',
    'seat_row',
    'seat_column',
    'nearest_amphi',
]

def final_drop(df):
    return df.drop(columns=[c for c in columns_to_drop_final if c in df.columns])

train = final_drop(train)
val   = final_drop(val)
test  = final_drop(test)

## 10. Final Data Quality Check

In [10]:
train['is_outside'] = train['is_outside'].astype(int)
test['is_outside'] = test['is_outside'].astype(int)
val['is_outside'] = val['is_outside'].astype(int)

for name, split in [('train', train), ('val', val), ('test', test)]:
    print(f"\n── {name} ({split.shape}) ─────────────────────")
    missing = split.isnull().sum()
    if missing.sum() > 0:
        print("Missing values:")
        print(missing[missing > 0])
    else:
        print("No missing values!")

print("\nData types:")
print(train.dtypes)

print("\nSummary statistics (train):")
print(train.describe())


── train ((4664, 26)) ─────────────────────
No missing values!

── val ((999, 26)) ─────────────────────
No missing values!

── test ((1000, 26)) ─────────────────────
No missing values!

Data types:
accuracy_mean         float64
is_outside              int64
sample_count            int64
dist_Amphi_1          float64
dist_Amphi_2          float64
dist_Amphi_3          float64
dist_Amphi_4          float64
dist_Amphi_5          float64
dist_Amphi_6          float64
dist_Amphi_7          float64
dist_Amphi_8          float64
dist_nearest          float64
dist_2nd              float64
dist_gap              float64
nearest_amphi_enc       int64
log_accuracy          float64
accuracy_bin            int64
high_accuracy_flag      int64
has_seat                int64
seat_block_enc          int64
seat_row_filled       float64
seat_column_filled    float64
seat_zone_id            int64
hour_sin              float64
hour_cos              float64
label_enc               int64
dtype: object

Summ

## 11. Export Data

In [11]:
output_dir = Path('../data/processed')

train.to_csv(output_dir / 'train_ready.csv', index=False)
val.to_csv(output_dir   / 'val_ready.csv',   index=False)
test.to_csv(output_dir  / 'test_ready.csv',  index=False)

for name, split in [('train_ready', train), ('val_ready', val), ('test_ready', test)]:
    size = split.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"  Saved {name}.csv  |  shape: {split.shape}  |  {size:.2f} MB")

print("\nFirst 5 rows (train):")
print(train.head())

  Saved train_ready.csv  |  shape: (4664, 26)  |  0.93 MB
  Saved val_ready.csv  |  shape: (999, 26)  |  0.20 MB
  Saved test_ready.csv  |  shape: (1000, 26)  |  0.20 MB

First 5 rows (train):
   accuracy_mean  is_outside  sample_count  dist_Amphi_1  dist_Amphi_2  \
0          7.899           0             1      0.000598      0.000383   
1         20.000           0             1      0.000211      0.000009   
2         34.320           0             1      0.000233      0.000021   
3         11.707           0             1      0.000204      0.000013   
4          9.671           0             1      0.000226      0.000013   

   dist_Amphi_3  dist_Amphi_4  dist_Amphi_5  dist_Amphi_6  dist_Amphi_7  ...  \
0      0.000164      0.000032      0.000578      0.000366      0.000186  ...   
1      0.000266      0.000377      0.000192      0.000023      0.000214  ...   
2      0.000246      0.000355      0.000214      0.000011      0.000193  ...   
3      0.000272      0.000384      0.00018